In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import time
import pinocchio as pin
from pinocchio.visualize import MeshcatVisualizer
from meshcat.transformations import translation_matrix, rotation_matrix
from meshcat.geometry import Box, MeshPhongMaterial, Cylinder, Sphere
from prm_motion_planner import MotionPlanner
from controller import Controller
from estimator import EKF

### Visualization

In [ ]:
# specify a path the the urdf files and meshes
urdf_model_path = "diffdrive.urdf"
mesh_dir = ""

# load the robot using pinocchio
robot = pin.RobotWrapper.BuildFromURDF(urdf_model_path, mesh_dir)

# vizualize the robot using meshcat
viz = MeshcatVisualizer(robot.model, robot.collision_model, robot.visual_model)
viz.initViewer(loadModel=True)

def show_robot(x,y,theta):
    quat = pin.Quaternion(pin.utils.rotate('z', theta)).coeffs()
    pos = np.array([x,y,0.1])

    viz.display(np.append(pos,quat))

In [ ]:
# define world
grid_size = 10
obstacle_radius = 0.2
min_val, max_val = -grid_size/2, grid_size/2

start = np.array([0.0, 0.0, 0.0]) # x, y, theta
goal = np.array([4.5, 4.0, 0.0])

In [ ]:
# Add a floor
# Add floor material
material_floor = MeshPhongMaterial()
material_floor.color = int(200) * 256**2 + int(200) * 256 + int(200)
# Add a floor
viz.viewer["/Floor"].set_object(
    Box([grid_size, grid_size, 0.01]),
    material_floor
)
viz.viewer["/Floor"].set_transform(
    translation_matrix([0, 0, -0.005])
)

material_goal = MeshPhongMaterial()
material_goal.color = int(255) * 256**2 + int(0) * 256 + int(0)

viz.viewer["/Goal"].set_object(
    Sphere(0.15),
    material_goal
)

viz.viewer["/Goal"].set_transform(
    translation_matrix([goal[0], goal[1], 0.15])
)

# Add obstacle material
material_obstacle = MeshPhongMaterial()
material_obstacle.color = int(100) * 256**2 + int(100) * 256 + int(100)

# Randomly generate 30 obstacle positions within a defined range
np.random.seed(6) #6
obstacle_positions = [
    np.array([np.random.uniform(-4.8, 4.8), np.random.uniform(-4.8, 4.8), 0.5])
    
    
    for _ in range(10)
]

# add cylinders for each obstacle
for i, pos in enumerate(obstacle_positions):
    viz.viewer[f"/Obstacle_{i}"].set_object(
        Cylinder(1, obstacle_radius), material_obstacle
    )
    T_world_obs = translation_matrix(pos)
    T_world_obs[:3, :3] = pin.utils.rotate('x', np.pi / 2)
    viz.viewer[f"/Obstacle_{i}"].set_transform(
        T_world_obs
    )

wall_obstacles = []
for i, pos in enumerate(obstacle_positions[::2]):
    # Connect a wall (box) between the two obstacles
    wall_length = np.linalg.norm(obstacle_positions[2*i][:2] - obstacle_positions[2*i+1][:2])
    wall_width = 2*obstacle_radius
    wall_height = 1.0
    wall_material = MeshPhongMaterial()
    wall_material.color = int(100) * 256**2 + int(100) * 256 + int(100)

    p1 = obstacle_positions[2*i][:2]
    p2 = obstacle_positions[2*i+1][:2]

    wall_position = (p1 + p2) / 2

    wall_obstacles.append({
        "p1": p1,
        "p2": p2,
        "wall_width": wall_width,
        "endpoint_radius": obstacle_radius
    })

    viz.viewer[f"/Wall_Obstacle_{i}"].set_object(
        Box([wall_length, wall_width, wall_height]), wall_material
    )
    # Set the wall rotation to align with the line between the two obstacles
    angle = np.arctan2(
        obstacle_positions[2*i+1][1] - obstacle_positions[2*i][1],
        obstacle_positions[2*i+1][0] - obstacle_positions[2*i][0]
    )
    viz.viewer[f"/Wall_Obstacle_{i}"].set_transform(
        translation_matrix([wall_position[0], wall_position[1], wall_height / 2]) @
        rotation_matrix(angle, [0, 0, 1])
    )

# Add walls around the floor
wall_thickness = 0.1
wall_height = 1.0

# Left wall
viz.viewer["/Wall_Left"].set_object(Box([wall_thickness, 10, wall_height]))
viz.viewer["/Wall_Left"].set_transform(
    translation_matrix([-5 - wall_thickness / 2, 0, wall_height / 2])
)

# Right wall
viz.viewer["/Wall_Right"].set_object(Box([wall_thickness, 10, wall_height]))
viz.viewer["/Wall_Right"].set_transform(
    translation_matrix([5 + wall_thickness / 2, 0, wall_height / 2])
)

# Front wall
viz.viewer["/Wall_Front"].set_object(Box([10, wall_thickness, wall_height]))
viz.viewer["/Wall_Front"].set_transform(
    translation_matrix([0, 5 + wall_thickness / 2, wall_height / 2])
)

# Back wall
viz.viewer["/Wall_Back"].set_object(Box([10, wall_thickness, wall_height]))
viz.viewer["/Wall_Back"].set_transform(
    translation_matrix([0, -5 - wall_thickness / 2, wall_height / 2])
)

# After defining wall_thickness, add boundary walls to wall_obstacles
half = grid_size / 2

boundary_walls = [
    # Left wall
    {"p1": np.array([-half, -half]), "p2": np.array([-half,  half]), "wall_width": wall_thickness, "endpoint_radius": 0.0},
    # Right wall
    {"p1": np.array([ half, -half]), "p2": np.array([ half,  half]), "wall_width": wall_thickness, "endpoint_radius": 0.0},
    # Front wall
    {"p1": np.array([-half,  half]), "p2": np.array([ half,  half]), "wall_width": wall_thickness, "endpoint_radius": 0.0},
    # Back wall
    {"p1": np.array([-half, -half]), "p2": np.array([ half, -half]), "wall_width": wall_thickness, "endpoint_radius": 0.0},
]

wall_obstacles_all = wall_obstacles + boundary_walls

# can you add a tower in each corner of the walls?
tower_height = 1.5
tower_radius = 0.3
tower_material = MeshPhongMaterial()
tower_material.color = int(100) * 256**2 + int(100) * 256 + int(100)

tower_positions = [
    np.array([-5 - wall_thickness / 2, 5 + wall_thickness / 2, tower_height / 2]),
    np.array([5 + wall_thickness / 2, 5 + wall_thickness / 2, tower_height / 2]),
    np.array([-5 - wall_thickness / 2, -5 - wall_thickness / 2, tower_height / 2]),
    np.array([5 + wall_thickness / 2, -5 - wall_thickness / 2, tower_height / 2])
]

for i, pos in enumerate(tower_positions):
    viz.viewer[f"/Tower_{i}"].set_object(
        Cylinder(tower_height, tower_radius), tower_material
    )
    T_world_tower = translation_matrix(pos)
    T_world_tower[:3, :3] = pin.utils.rotate('x', np.pi / 2)
    viz.viewer[f"/Tower_{i}"].set_transform(
        T_world_tower
    )

In [ ]:
def random_goal(obstacle_positions, obstacle_radius, grid_size=10, min_clearance=0.5, max_attempts=100):
    """Generate a random goal position that doesn't collide with obstacles or walls."""
    margin = grid_size / 2 - 0.5  # stay away from walls
    for _ in range(max_attempts):
        x = np.random.uniform(-margin, margin)
        y = np.random.uniform(-margin, margin)
        theta = np.random.uniform(-np.pi, np.pi)
        
        # Check clearance from all obstacles
        pos = np.array([x, y])
        too_close = any(
            np.linalg.norm(pos - obs[:2]) < obstacle_radius + min_clearance
            for obs in obstacle_positions
        )
        # Also keep away from start
        too_close = too_close or np.linalg.norm(pos) < min_clearance

        if not too_close:
            return np.array([x, y, theta])
    
    raise ValueError("Could not find a valid goal after max attempts — try reducing min_clearance.")


def update_goal_visual(viz, goal):
    """Move the goal sphere in the visualizer."""
    viz.viewer["/Goal"].set_transform(
        translation_matrix([goal[0], goal[1], 0.15])
    )


# --- Usage ---
goal = random_goal(obstacle_positions, obstacle_radius)
update_goal_visual(viz, goal)
print(f"New goal: {goal}")

In [ ]:
x,y,theta = start
show_robot(x,y,theta)

## Motion Planning

In [ ]:
planner = MotionPlanner(
    start=start,
    goal=goal,
    obstacles=wall_obstacles_all,
    grid_size=grid_size,
    obstacle_radius=obstacle_radius,
    robot_radius=0.35
)

if planner.is_collision(goal):
    raise ValueError("Goal is in collision! Please choose a different goal position.")

path = planner.global_planner(N=1000, k=10)

### Simulation

In [6]:
# Global constants for bicycle model (Task 4)"
WHEELBASE = 0.4         # distance between front and rear axle (meters)
MAX_STEER_DEG = 35.0    # max steering angle (degrees)

In [ ]:
# Define robot dynamics
def discrete_dynamics(state, control_input, dt, model_mismatch=False):
    """
    Update the robot's state based on its dynamics.

    Parameters:
    - state: Current state [x, y, theta]
    - control_input: Control input [linear_velocity, angular_velocity]
    - dt: Time step

    Returns:
    - Updated state [x, y, theta]
    """
    x, y, theta = state
    linear_velocity, angular_velocity = control_input
    if model_mismatch:
        # Introduce model mismatch by adding noise to the control input
        linear_velocity += np.random.normal(0, 0.05) # Noise (0, 1.0)
        angular_velocity += np.random.normal(0, 0.05)

    # Update state using differential drive kinematics
    x += linear_velocity * np.cos(theta) * dt
    y += linear_velocity * np.sin(theta) * dt
    theta += angular_velocity * dt

    # Normalize theta to keep it within [-pi, pi]
    theta = np.arctan2(np.sin(theta), np.cos(theta))

    return np.array([x, y, theta])


In [11]:
# initialize EKF and choose model type (Task 4)
ekf = EKF(x0=start[0], y0=start[1], theta0=start[2], model='bicycle')

In [ ]:
# New: working!

def simulation(controller, path):
    dt = 0.01
    simulation_time = 90
    desired_speed = 0.16
    q = start.copy()

    # Build a time-parameterized reference by arc length
    # Compute cumulative distances along path
    dists = [0.0]
    for i in range(1, len(path)):
        dists.append(dists[-1] + np.linalg.norm(path[i][:2] - path[i-1][:2]))
    total_length = dists[-1]

    def get_reference_at_arc(s):
        s = np.clip(s, 0, total_length)
        for i in range(len(path) - 1):
            if s <= dists[i+1] or i == len(path) - 2:
                seg_len = dists[i+1] - dists[i]
                if seg_len < 1e-6:
                    continue
                alpha = np.clip((s - dists[i]) / seg_len, 0, 1)
                p = path[i][:2] + alpha * (path[i+1][:2] - path[i][:2])
                seg_vec = path[i+1][:2] - path[i][:2]

                # On the final segment, blend toward the goal orientation
                if i == len(path) - 2:
                    theta_tangent = np.arctan2(seg_vec[1], seg_vec[0])
                    goal_theta = path[-1][2] if len(path[-1]) > 2 else theta_tangent
                    theta_d = theta_tangent + alpha * controller.wrap_to_pi(goal_theta - theta_tangent)
                else:
                    theta_d = np.arctan2(seg_vec[1], seg_vec[0])

                v_d = desired_speed if s < total_length else 0.0
                return np.array([p[0], p[1], theta_d]), v_d, 0.0

        # Past end of path: hold position, rotate to goal theta
        goal_theta = path[-1][2] if len(path[-1]) > 2 else 0.0
        return np.array([path[-1][0], path[-1][1], goal_theta]), 0.0, 0.0

    # Virtual reference advances at desired_speed in time
    s_ref = 0.0

    for t in np.arange(0, simulation_time, dt):
        # Advance the virtual reference point forward in time
        if s_ref < total_length:
            s_ref += desired_speed * dt

        q_d, v_d, w_d = get_reference_at_arc(s_ref)

        # Eq. 13.31 controller
        v, w = controller.non_lin_fb_controller(q, q_d, v_d, w_d)
        q = discrete_dynamics(q, [v, w], dt, model_mismatch=False)

        _, _, phi_e = controller.error_coordinates(q, q_d)
        print(f"t={t:.2f} s={s_ref:.2f}/{total_length:.2f} q={np.round(q,3)} "
              f"q_d={np.round(q_d,3)} v={v:.3f} w={w:.3f} phi_e={phi_e:.3f}")
        show_robot(q[0], q[1], q[2])

        goal_theta = path[-1][2] if len(path[-1]) > 2 else 0.0

        if s_ref >= total_length and np.linalg.norm(q[:2] - path[-1][:2]) < 0.3 and \
            np.abs(controller.wrap_to_pi(q[2] - goal_theta)) < 0.1:
            print(f"Goal reached at t={t:.2f}!")
            break

In [ ]:
controller = Controller(
    K1 = 1.0,
    K2 = 35.0,
    K3 = 18.00,
    wheelbase = WHEELBASE,
    max_steer_deg = MAX_STEER_DEG # expects degrees, will convert to radians internally
)

simulation(controller, path=path)

In [ ]:
print(f"Path length: {len(path)}")
print(f"First waypoint: {path[0]}")
print(f"Second waypoint: {path[1]}")
print(f"Last waypoint: {path[-1]}")
print(f"Goal: {goal}")

print(path[:3])
print(path[-1])

In [ ]:
# --- EKF standalone test (used for NEES/NIS validation) ---

# create storage
history = {'gps_x': [], 'gps_y': [], 
           'ekf_x': [], 'ekf_y': [], 'ekf_theta': [],
           'true_x': [], 'true_y': [], 'true_theta': [],
           'P': [], 'S': []}

def controller_with_ekf(x, y, theta, z):
    # controller currently uses true values, later use x_est, y_est, theta_est
    # EKF predict step uses last control input
    # we use fixed inputs here same as dummy controller
    u, w = 2, -2
    
    ekf.predict(u, w, dt)  # predict with control inputs
    ekf.update(z)          # correct with GPS measurement
    
    # Use EKF estimate instead of raw state
    x_est, y_est, theta_est = ekf.get_state()

    # store history
    history['gps_x'].append(z[0])
    history['gps_y'].append(z[1])
    history['ekf_x'].append(x_est)
    history['ekf_y'].append(y_est)
    history['ekf_theta'].append(theta_est)
    history['true_x'].append(x)
    history['true_y'].append(y)
    history['true_theta'].append(theta)
    history['P'].append(ekf.P.copy())
    history['S'].append(ekf.S.copy())

    # # ----------------------------------------------
    # # check
    # if not hasattr(controller_with_ekf, 'step'):
    #     controller_with_ekf.step = 0
    # controller_with_ekf.step += 1
    
    # if controller_with_ekf.step % 100 == 0:
    #     print(f"GPS raw:  x={z[0]:.3f}, y={z[1]:.3f}")
    #     print(f"EKF est: x={x_est:.3f}, y={y_est:.3f}, theta={theta_est:.3f}")
    #     print(f"True:     x={x:.3f}, y={y:.3f}, theta={theta:.3f}")
    #     print("---")
    # # ----------------------------------------------

    return u, w


In [ ]:
simulation(controller_with_ekf)

In [ ]:
# -----------------------------------------------------------------
# plot
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['gps_x'], history['gps_y'], 'r.', alpha=0.3, markersize=2, label='GPS raw')
axes[0].plot(history['ekf_x'], history['ekf_y'], 'b-', linewidth=1.5, label='EKF estimate')
axes[0].plot(history['true_x'], history['true_y'], 'g-', linewidth=1.5, label='True')
axes[0].set_title('XY trajectory')
axes[0].legend()
axes[0].set_aspect('equal')

axes[1].plot(history['true_theta'], 'g-', label='True theta')
axes[1].plot(history['ekf_theta'], 'b-', label='EKF theta')
axes[1].set_title('Theta over time')
axes[1].legend()

# position error
ekf_err = np.sqrt((np.array(history['ekf_x']) - np.array(history['true_x']))**2 + 
                  (np.array(history['ekf_y']) - np.array(history['true_y']))**2)
gps_err = np.sqrt((np.array(history['gps_x']) - np.array(history['true_x']))**2 + 
                  (np.array(history['gps_y']) - np.array(history['true_y']))**2)

axes[2].plot(gps_err, 'r-', alpha=0.5, label='GPS error')
axes[2].plot(ekf_err, 'b-', label='EKF error')
axes[2].set_title('Position error over time')
axes[2].legend()

plt.tight_layout()
plt.show()
# -----------------------------------------------------------------

In [ ]:
from scipy.stats import chi2

# -------------------------------------------------------------------------------
# compute NEES and NIS
nees_values = []
nis_values  = []

for i in range(len(history['true_x'])):
    # --- NEES ---
    x_true = np.array([history['true_x'][i], 
                       history['true_y'][i], 
                       history['true_theta'][i]])
    x_est  = np.array([history['ekf_x'][i], 
                       history['ekf_y'][i], 
                       history['ekf_theta'][i]])
    P      = history['P'][i]          # covariance at each step (we need to store this)
    
    x_tilde = x_true - x_est
    # normalize theta error to [-pi, pi]
    x_tilde[2] = np.arctan2(np.sin(x_tilde[2]), np.cos(x_tilde[2]))
    
    nees = x_tilde @ np.linalg.inv(P) @ x_tilde
    nees_values.append(nees)
    
    # --- NIS ---
    z     = np.array([history['gps_x'][i], history['gps_y'][i]])
    x_bar = np.array([history['ekf_x'][i], history['ekf_y'][i]])  
    # innovation
    nu    = z - x_bar   # H @ x_est = x_est[:2] since H picks first two states
    S     = history['S'][i]           # innovation covariance at each step
    
    nis = nu @ np.linalg.inv(S) @ nu
    nis_values.append(nis)

nees_values = np.array(nees_values)
nis_values  = np.array(nis_values)

# chi-squared bounds 
# 95% confidence interval
n = 3   # state dimensions  -> NEES ~ chi2(3)
m = 2   # measurement dimensions -> NIS ~ chi2(2)

nees_lower = chi2.ppf(0.025, df=n)   # 2.5th percentile
nees_upper = chi2.ppf(0.975, df=n)   # 97.5th percentile
nis_lower  = chi2.ppf(0.025, df=m)
nis_upper  = chi2.ppf(0.975, df=m)

# plot
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(nees_values, 'b-', alpha=0.6, linewidth=0.8, label='NEES')
axes[0].axhline(nees_upper, color='r', linestyle='--', label=f'95% upper bound ({nees_upper:.2f})')
axes[0].axhline(nees_lower, color='r', linestyle='--', label=f'95% lower bound ({nees_lower:.2f})')
axes[0].axhline(n, color='g', linestyle='-', alpha=0.5, label=f'Expected mean ({n})')
axes[0].set_title('NEES — should stay within red bounds 95% of the time')
axes[0].set_ylabel('NEES')
axes[0].set_xlabel('timestep')
axes[0].legend()
axes[0].set_ylim(0, 20)

axes[1].plot(nis_values, 'b-', alpha=0.6, linewidth=0.8, label='NIS')
axes[1].axhline(nis_upper, color='r', linestyle='--', label=f'95% upper bound ({nis_upper:.2f})')
axes[1].axhline(nis_lower, color='r', linestyle='--', label=f'95% lower bound ({nis_lower:.2f})')
axes[1].axhline(m, color='g', linestyle='-', alpha=0.5, label=f'Expected mean ({m})')
axes[1].set_title('NIS — should stay within red bounds 95% of the time')
axes[1].set_ylabel('NIS')
axes[1].set_xlabel('timestep')
axes[1].legend()
axes[1].set_ylim(0, 15)

plt.tight_layout()
plt.show()

# print summary
pct_nees_in = np.mean((nees_values >= nees_lower) & (nees_values <= nees_upper)) * 100
pct_nis_in  = np.mean((nis_values  >= nis_lower)  & (nis_values  <= nis_upper))  * 100

print(f"NEES: {pct_nees_in:.1f}% of values within 95% bounds (expect ~95%)")
print(f"NIS:  {pct_nis_in:.1f}% of values within 95% bounds (expect ~95%)")
print(f"Mean NEES: {nees_values.mean():.2f} (expect ~{n})")
print(f"Mean NIS:  {nis_values.mean():.2f}  (expect ~{m})")
# -------------------------------------------------------------------------------

### Car-like robot

In [2]:
# specify a path the the urdf files and meshes
urdf_model_path = "carlike.urdf"
mesh_dir = ""

# load the robot using pinocchio
robot = robot = pin.RobotWrapper.BuildFromURDF(
    urdf_model_path,
    mesh_dir,
    pin.JointModelFreeFlyer()
)

# vizualize the robot using meshcat
viz = MeshcatVisualizer(robot.model, robot.collision_model, robot.visual_model)
viz.initViewer(loadModel=True)

def show_robot(x, y, theta, delta=0.0):
    q_vis = np.zeros(robot.model.nq)

    # Free-flyer base position
    q_vis[0:3] = np.array([x, y, 0.1])

    # Free-flyer base orientation
    quat = pin.Quaternion(pin.utils.rotate('z', theta)).coeffs()
    q_vis[3:7] = quat

    # Front steering joints
    left_id = robot.model.getJointId("front_left_steer_joint")
    right_id = robot.model.getJointId("front_right_steer_joint")

    left_idx = robot.model.joints[left_id].idx_q
    right_idx = robot.model.joints[right_id].idx_q

    q_vis[left_idx] = delta
    q_vis[right_idx] = delta

    viz.display(q_vis)

# define world
grid_size = 10
obstacle_radius = 0.2
wheelbase = 0.4
max_steer_deg = 35.0
min_val, max_val = -grid_size/2, grid_size/2

start = np.array([0.0, 0.0, 0.0]) # x, y, theta
goal = np.array([4.5, 4.0, 0.0])

# Add a floor
# Add floor material
material_floor = MeshPhongMaterial()
material_floor.color = int(200) * 256**2 + int(200) * 256 + int(200)
# Add a floor
viz.viewer["/Floor"].set_object(
    Box([grid_size, grid_size, 0.01]),
    material_floor
)
viz.viewer["/Floor"].set_transform(
    translation_matrix([0, 0, -0.005])
)

material_goal = MeshPhongMaterial()
material_goal.color = int(255) * 256**2 + int(0) * 256 + int(0)

viz.viewer["/Goal"].set_object(
    Sphere(0.15),
    material_goal
)

viz.viewer["/Goal"].set_transform(
    translation_matrix([goal[0], goal[1], 0.15])
)

# Add obstacle material
material_obstacle = MeshPhongMaterial()
material_obstacle.color = int(100) * 256**2 + int(100) * 256 + int(100)

# Randomly generate 30 obstacle positions within a defined range
np.random.seed(6) #6
obstacle_positions = [
    np.array([np.random.uniform(-4.8, 4.8), np.random.uniform(-4.8, 4.8), 0.5])
    
    
    for _ in range(10)
]

# add cylinders for each obstacle
for i, pos in enumerate(obstacle_positions):
    viz.viewer[f"/Obstacle_{i}"].set_object(
        Cylinder(1, obstacle_radius), material_obstacle
    )
    T_world_obs = translation_matrix(pos)
    T_world_obs[:3, :3] = pin.utils.rotate('x', np.pi / 2)
    viz.viewer[f"/Obstacle_{i}"].set_transform(
        T_world_obs
    )

wall_obstacles = []
for i, pos in enumerate(obstacle_positions[::2]):
    # Connect a wall (box) between the two obstacles
    wall_length = np.linalg.norm(obstacle_positions[2*i][:2] - obstacle_positions[2*i+1][:2])
    wall_width = 2*obstacle_radius
    wall_height = 1.0
    wall_material = MeshPhongMaterial()
    wall_material.color = int(100) * 256**2 + int(100) * 256 + int(100)

    p1 = obstacle_positions[2*i][:2]
    p2 = obstacle_positions[2*i+1][:2]

    wall_position = (p1 + p2) / 2

    wall_obstacles.append({
        "p1": p1,
        "p2": p2,
        "wall_width": wall_width,
        "endpoint_radius": obstacle_radius
    })

    viz.viewer[f"/Wall_Obstacle_{i}"].set_object(
        Box([wall_length, wall_width, wall_height]), wall_material
    )
    # Set the wall rotation to align with the line between the two obstacles
    angle = np.arctan2(
        obstacle_positions[2*i+1][1] - obstacle_positions[2*i][1],
        obstacle_positions[2*i+1][0] - obstacle_positions[2*i][0]
    )
    viz.viewer[f"/Wall_Obstacle_{i}"].set_transform(
        translation_matrix([wall_position[0], wall_position[1], wall_height / 2]) @
        rotation_matrix(angle, [0, 0, 1])
    )

# Add walls around the floor
wall_thickness = 0.1
wall_height = 1.0

# Left wall
viz.viewer["/Wall_Left"].set_object(Box([wall_thickness, 10, wall_height]))
viz.viewer["/Wall_Left"].set_transform(
    translation_matrix([-5 - wall_thickness / 2, 0, wall_height / 2])
)

# Right wall
viz.viewer["/Wall_Right"].set_object(Box([wall_thickness, 10, wall_height]))
viz.viewer["/Wall_Right"].set_transform(
    translation_matrix([5 + wall_thickness / 2, 0, wall_height / 2])
)

# Front wall
viz.viewer["/Wall_Front"].set_object(Box([10, wall_thickness, wall_height]))
viz.viewer["/Wall_Front"].set_transform(
    translation_matrix([0, 5 + wall_thickness / 2, wall_height / 2])
)

# Back wall
viz.viewer["/Wall_Back"].set_object(Box([10, wall_thickness, wall_height]))
viz.viewer["/Wall_Back"].set_transform(
    translation_matrix([0, -5 - wall_thickness / 2, wall_height / 2])
)

# After defining wall_thickness, add boundary walls to wall_obstacles
half = grid_size / 2

boundary_walls = [
    # Left wall
    {"p1": np.array([-half, -half]), "p2": np.array([-half,  half]), "wall_width": wall_thickness, "endpoint_radius": 0.0},
    # Right wall
    {"p1": np.array([ half, -half]), "p2": np.array([ half,  half]), "wall_width": wall_thickness, "endpoint_radius": 0.0},
    # Front wall
    {"p1": np.array([-half,  half]), "p2": np.array([ half,  half]), "wall_width": wall_thickness, "endpoint_radius": 0.0},
    # Back wall
    {"p1": np.array([-half, -half]), "p2": np.array([ half, -half]), "wall_width": wall_thickness, "endpoint_radius": 0.0},
]

wall_obstacles_all = wall_obstacles + boundary_walls

# can you add a tower in each corner of the walls?
tower_height = 1.5
tower_radius = 0.3
tower_material = MeshPhongMaterial()
tower_material.color = int(100) * 256**2 + int(100) * 256 + int(100)

tower_positions = [
    np.array([-5 - wall_thickness / 2, 5 + wall_thickness / 2, tower_height / 2]),
    np.array([5 + wall_thickness / 2, 5 + wall_thickness / 2, tower_height / 2]),
    np.array([-5 - wall_thickness / 2, -5 - wall_thickness / 2, tower_height / 2]),
    np.array([5 + wall_thickness / 2, -5 - wall_thickness / 2, tower_height / 2])
]

for i, pos in enumerate(tower_positions):
    viz.viewer[f"/Tower_{i}"].set_object(
        Cylinder(tower_height, tower_radius), tower_material
    )
    T_world_tower = translation_matrix(pos)
    T_world_tower[:3, :3] = pin.utils.rotate('x', np.pi / 2)
    viz.viewer[f"/Tower_{i}"].set_transform(
        T_world_tower
    )

x, y, theta = start
show_robot(x, y, theta, delta=0.0)

You can open the visualizer by visiting the following URL:
http://127.0.0.1:7002/static/


In [3]:
def random_goal(obstacle_positions, obstacle_radius, grid_size=10, min_clearance=0.5, max_attempts=100):
    """Generate a random goal position that doesn't collide with obstacles or walls."""
    margin = grid_size / 2 - 0.5  # stay away from walls
    for _ in range(max_attempts):
        x = np.random.uniform(-margin, margin)
        y = np.random.uniform(-margin, margin)
        theta = np.random.uniform(-np.pi, np.pi)
        
        # Check clearance from all obstacles
        pos = np.array([x, y])
        too_close = any(
            np.linalg.norm(pos - obs[:2]) < obstacle_radius + min_clearance
            for obs in obstacle_positions
        )
        # Also keep away from start
        too_close = too_close or np.linalg.norm(pos) < min_clearance

        if not too_close:
            return np.array([x, y, theta])
    
    raise ValueError("Could not find a valid goal after max attempts — try reducing min_clearance.")


def update_goal_visual(viz, goal):
    """Move the goal sphere in the visualizer."""
    viz.viewer["/Goal"].set_transform(
        translation_matrix([goal[0], goal[1], 0.15])
    )


# --- Usage ---
goal = random_goal(obstacle_positions, obstacle_radius)
update_goal_visual(viz, goal)
print(f"New goal: {goal}")

New goal: [-4.00972943  1.96773513  1.89859364]


In [7]:
# Bicycle dynamics for car-like robot (Task 4)
def discrete_dynamics(state, control_input, dt, wheelbase=WHEELBASE, model_mismatch=False):
    """
    Bicycle model kinematics for a car-like robot.
    state:         [x, y, theta]
    control_input: [v, delta]   (linear speed, steering angle)
    wheelbase:     L (distance between front and rear axle)
    """
    x, y, theta = state
    v, delta = control_input

    # Clamp steering angle to physical limits
    delta_max = np.radians(35)           # tune to your robot
    delta = np.clip(delta, -delta_max, delta_max)

    if model_mismatch:
        v     += np.random.normal(0, 0.05)
        delta += np.random.normal(0, 0.02)

    # Bicycle model update
    x     += v * np.cos(theta) * dt
    y     += v * np.sin(theta) * dt
    theta += (v * np.tan(delta) / wheelbase) * dt
    theta  = np.arctan2(np.sin(theta), np.cos(theta))   # wrap to [-π, π]

    return np.array([x, y, theta])

In [8]:
# New: working!

def simulation(controller, path):
    dt = 0.01
    simulation_time = 90
    desired_speed = 0.2
    q = start.copy()

    # Build a time-parameterized reference by arc length
    # Compute cumulative distances along path
    dists = [0.0]
    for i in range(1, len(path)):
        dists.append(dists[-1] + np.linalg.norm(path[i][:2] - path[i-1][:2]))
    total_length = dists[-1]

    def get_reference_at_arc(s):
        s = np.clip(s, 0, total_length)
        for i in range(len(path) - 1):
            if s <= dists[i+1] or i == len(path) - 2:
                seg_len = dists[i+1] - dists[i]
                if seg_len < 1e-6:
                    continue
                alpha = np.clip((s - dists[i]) / seg_len, 0, 1)
                p = path[i][:2] + alpha * (path[i+1][:2] - path[i][:2])
                seg_vec = path[i+1][:2] - path[i][:2]

                # On the final segment, blend toward the goal orientation
                if i == len(path) - 2:
                    theta_tangent = np.arctan2(seg_vec[1], seg_vec[0])
                    goal_theta = path[-1][2] if len(path[-1]) > 2 else theta_tangent
                    theta_d = theta_tangent + alpha * controller.wrap_to_pi(goal_theta - theta_tangent)
                else:
                    theta_d = np.arctan2(seg_vec[1], seg_vec[0])

                v_d = desired_speed if s < total_length else 0.0
                return np.array([p[0], p[1], theta_d]), v_d, 0.0
        
        # Past end of path: hold position, rotate to goal theta
        goal_theta = path[-1][2] if len(path[-1]) > 2 else 0.0
        return np.array([path[-1][0], path[-1][1], goal_theta]), 0.0, 0.0

    # Virtual reference advances at desired_speed in time
    s_ref = 0.0

    # for bicycle model: track steering angle and its rate (delta_dot) for EKF
    # EKF.predict expects (v, delta_dot), not (v, omega)!
    v_prev = 0.0
    delta_prev = 0.0
    delta_dot_prev = 0.0  # steering angle velocity

    for t in np.arange(0, simulation_time, dt):
        # Advance the virtual reference point forward in time
        if s_ref < total_length:
            s_ref += desired_speed * dt

        q_d, v_d, w_d = get_reference_at_arc(s_ref)

        # GPS measurement
        z = np.array([q[0], q[1]]) + np.random.normal(0, 0.03, 2)
        
        # EKF: predict with previous control inputs (converted to steering-angle velocity)
        # For bicycle model, EKF expects steering angle velocity, not omega!
        ekf.predict(v_prev, delta_dot_prev, dt, wheelbase=WHEELBASE)
        ekf.update(z)
        q_est = ekf.get_state()

        # Eq. 13.31 controller (returns v, omega where omega is angular velocity)
        v, delta = controller.car_controller(q_est, q_d, v_d, w_d)

        # update true state with discrete_dynamics using [v, delta]
        q = discrete_dynamics(q, [v, delta], dt, model_mismatch=False)

        # compute steering angle velocity for next EKF predict step
        delta_dot = (delta - delta_prev) / dt

        # save the control used for the next EKF prediction step
        v_prev = v
        delta_prev = delta
        delta_dot_prev = delta_dot

        _, _, phi_e = controller.error_coordinates(q_est, q_d) # error compared to estimation?
        print(f"t={t:.2f} s={s_ref:.2f}/{total_length:.2f} q={np.round(q_est,3)} "
              f"q_d={np.round(q_d,3)} v={v:.3f} delta={np.degrees(delta):.1f}° phi_e={phi_e:.3f}")
        show_robot(q[0], q[1], q[2])

        goal_theta = path[-1][2] if len(path[-1]) > 2 else 0.0

        pos_error = np.linalg.norm(q[:2] - path[-1][:2])
        theta_error = np.abs(controller.wrap_to_pi(q[2] - goal_theta))

        if s_ref >= total_length and pos_error < 0.2 and theta_error < 0.05:
            print(f"Goal reached at t={t:.2f}!")
            print(f"Position error: {pos_error:.3f}")
            print(f"Theta error: {theta_error:.3f}")
            break

In [ ]:
%pip install dubins
from car_motion_planner import CarMotionPlanner

planner = CarMotionPlanner(
    start=start,
    goal=goal,
    obstacles=wall_obstacles_all,
    grid_size=grid_size,
    obstacle_radius=obstacle_radius,
    robot_radius=0.30
)

if planner.is_collision(goal):
    raise ValueError("Goal is in collision! Please choose a different goal position.")

path = planner.global_planner(N=1000, k=10)

ModuleNotFoundError: No module named 'dubins'

In [ ]:
controller = Controller(
    K1 = 1.0,
    K2 = 35.0,
    K3 = 18.00,
    wheelbase = WHEELBASE,
    max_steer_deg = MAX_STEER_DEG # expects degrees, will convert to radians internally
)

simulation(controller, path=path)

t=0.00 s=0.00/7.22 q=[-4.051  2.246  1.9  ] q_d=[-2.000e-03  1.000e-03  2.426e+00] v=0.000 delta=0.0° phi_e=-0.526
t=0.01 s=0.00/7.22 q=[-4.043  2.225  1.898] q_d=[-0.003  0.003  2.426] v=0.000 delta=0.0° phi_e=-0.529
t=0.02 s=0.01/7.22 q=[-4.035  2.203  1.895] q_d=[-0.005  0.004  2.426] v=0.000 delta=0.0° phi_e=-0.531
t=0.03 s=0.01/7.22 q=[-4.027  2.183  1.893] q_d=[-0.006  0.005  2.426] v=0.000 delta=0.0° phi_e=-0.534
t=0.04 s=0.01/7.22 q=[-4.019  2.162  1.89 ] q_d=[-0.008  0.007  2.426] v=0.000 delta=0.0° phi_e=-0.536
t=0.05 s=0.01/7.22 q=[-4.011  2.141  1.887] q_d=[-0.009  0.008  2.426] v=0.000 delta=0.0° phi_e=-0.539
t=0.06 s=0.01/7.22 q=[-4.004  2.12   1.885] q_d=[-0.011  0.009  2.426] v=0.000 delta=0.0° phi_e=-0.541
t=0.07 s=0.02/7.22 q=[-3.996  2.099  1.882] q_d=[-0.012  0.01   2.426] v=0.000 delta=0.0° phi_e=-0.544
t=0.08 s=0.02/7.22 q=[-3.988  2.079  1.88 ] q_d=[-0.014  0.012  2.426] v=0.000 delta=0.0° phi_e=-0.547
t=0.09 s=0.02/7.22 q=[-3.981  2.059  1.877] q_d=[-0.015  0.01